In [1]:
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from dotenv import load_dotenv

2026-03-20 11:52:34.001913: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 11:52:34.042420: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 11:52:35.555627: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 11:52:35.556076: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not 

In [2]:
notebook_login()

In [3]:
dataset = load_dataset('hate-speech-portuguese/hate_speech_portuguese',split='train[:10%]')

In [4]:
dataset

Dataset({
    features: ['text', 'label', 'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'],
    num_rows: 567
})

In [5]:
dataset= dataset.remove_columns(['hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'])
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 567
})

In [6]:
dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 453
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 114
    })
})

In [7]:
dataset['train'][0]

{'text': '@amandassini medir minhas palavras? agora além de silenciar a angelica vc quer me silenciar tambem?',
 'label': 0}

In [8]:
checkpoint = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example['text'],padding=True,truncation=True)

/home/orlandojunior/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
tokenizer_datasets = dataset.map(tokenize_function, batched=True)
data_collor = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments

output_dir = './bert/hate-speech-test'
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100, # a cada 100 steps ele vai salvar o log
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

In [12]:
from transformers import Trainer, AutoModelForSequenceClassification
import os

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
os.environ["WANDB_DISABLE"] = "true"
os.environ["WANDB_MODE"] = "offline"

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [17]:
trainer = Trainer(
    model,
    training_args,
    train_dataset= tokenizer_datasets['train'],
    eval_dataset= tokenizer_datasets['test'],
    data_collator=data_collor,
    tokenizer=tokenizer,
    compute_metrics=compute_metric
)

In [18]:
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=171, training_loss=0.6058594553094161, metrics={'train_runtime': 18.1286, 'train_samples_per_second': 74.964, 'train_steps_per_second': 9.433, 'total_flos': 46093154081436.0, 'train_loss': 0.6058594553094161, 'epoch': 3.0})

In [19]:
trainer.evaluate()

{'eval_loss': 0.5418471693992615,
 'eval_accuracy': 0.7280701754385965,
 'eval_runtime': 0.5523,
 'eval_samples_per_second': 206.421,
 'eval_steps_per_second': 27.161,
 'epoch': 3.0}

In [20]:
trainer.save_model()

In [21]:
trainer.push_to_hub("Orlandovpjunior/modelhate")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.52k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Orlandovpjunior/hate-speech-test/commit/a487f1147e68f0b666f821feec471b0fba1f1eae', commit_message='Orlandovpjunior/modelhate', commit_description='', oid='a487f1147e68f0b666f821feec471b0fba1f1eae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Orlandovpjunior/hate-speech-test', endpoint='https://huggingface.co', repo_type='model', repo_id='Orlandovpjunior/hate-speech-test'), pr_revision=None, pr_num=None)

In [22]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="Orlandovpjunior/hate-speech-test")

config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [24]:
pipe("Cara vai me censurar")

[{'label': 'LABEL_1', 'score': 0.5222259163856506}]